# **Sales Summary**

## **Revenue by Region**

In [0]:
%run "/Workspace/capstone_project/capstone_project_cyntexa/capstone_bundle/src/gold/product_analytics"

In [0]:
# %sql
# create or replace view identifier(:catalog).gold.sales_by_region as
# SELECT
#     region,
#     round(SUM(sale_amount), 2) AS total_revenue
# FROM identifier(:catalog).silver.sales_clean
# GROUP BY region;

## **Revenue by Category**

In [0]:
%sql
create or replace view identifier(:catalog).gold.sales_by_category as
SELECT
    p.category,
    round(SUM(s.total_amount), 2) AS total_revenue
FROM identifier(:catalog).silver.sales_clean s
JOIN identifier(:catalog).silver.products_scd2 p
ON s.product_id = p.product_id
WHERE p.is_current = true
GROUP BY p.category;

## **Revenue by time period**

In [0]:
%sql
CREATE OR REPLACE VIEW identifier(:catalog).gold.sales_by_time_period as
SELECT
    YEAR(sale_date) AS year,
    MONTH(sale_date) AS month,
    round(SUM(total_amount), 2) AS total_revenue
FROM identifier(:catalog).silver.sales_clean
GROUP BY
YEAR(sale_date),
MONTH(sale_date)
order by year asc, month asc;

## **Year-over-year growth rates**

In [0]:
%sql
create or replace view identifier(:catalog).gold.yoy_growth_rate as
WITH yearly_sales AS (

    SELECT
        YEAR(sale_date) AS sales_year,
        SUM(total_amount) AS total_revenue
    FROM identifier(:catalog).silver.sales_clean
    GROUP BY YEAR(sale_date)

),

yoy_growth AS (

    SELECT
        sales_year,
        total_revenue,

        LAG(total_revenue) OVER (
            ORDER BY sales_year
        ) AS previous_year_revenue

    FROM yearly_sales

)

SELECT

    sales_year,

    ROUND(total_revenue, 2) AS total_revenue,

    ROUND(previous_year_revenue, 2) AS previous_year_revenue,

    ROUND(
        (
            (total_revenue - previous_year_revenue)
            /  NULLIF(previous_year_revenue, 0)
        ) * 100,
        2
    ) AS yoy_growth_percentage

FROM yoy_growth

ORDER BY sales_year;

## **Top 10 customers by revenue**

In [0]:
%sql
create or replace view identifier(:catalog).gold.top_customers as
WITH customer_revenue AS (
    SELECT
        c.customer_id,
        c.name AS customer_name,
        SUM(s.total_amount) AS total_revenue
    FROM identifier(:catalog).silver.sales_clean s
    JOIN identifier(:catalog).silver.customers_clean c
        ON s.customer_id = c.customer_id
    GROUP BY
        c.customer_id,
        c.name

),

customer_ranking AS (
    SELECT
        customer_id,
        customer_name,
        ROUND(total_revenue, 2) AS total_revenue,

        ROW_NUMBER() OVER (
            ORDER BY total_revenue DESC
        ) AS row_number,

        RANK() OVER (
            ORDER BY total_revenue DESC
        ) AS rank,

        DENSE_RANK() OVER (
            ORDER BY total_revenue DESC
        ) AS dense_rank

    FROM customer_revenue

)

SELECT *
FROM customer_ranking
WHERE row_number <= 10
ORDER BY row_number;

In [0]:
%sql
create or replace view identifier(:catalog).gold.revenue_by_state as
SELECT
    c.state,
    ROUND(SUM(s.total_amount), 2) AS total_revenue
FROM identifier(:catalog).silver.sales_clean s
JOIN identifier(:catalog).silver.customers_clean c
    ON s.customer_id = c.customer_id
GROUP BY c.state
ORDER BY total_revenue DESC;

In [0]:
%sql
create or replace view identifier(:catalog).gold.customer_count_by_state as
SELECT
    state,
    COUNT(*) AS total_customers
FROM identifier(:catalog).silver.customers_clean
GROUP BY state
ORDER BY total_customers DESC;

In [0]:
%sql
create or replace view identifier(:catalog).gold.revenue_by_city as
SELECT
    c.city,
    ROUND(SUM(s.total_amount), 2) AS total_revenue
FROM identifier(:catalog).silver.sales_clean s
JOIN identifier(:catalog).silver.customers_clean c
    ON s.customer_id = c.customer_id
GROUP BY c.city
ORDER BY total_revenue DESC;